# Day11 LangChain 与 LangSmith

本课对应网盘目录 `2、AI智能体测试/day11-Langchain与langsmith`，与前面 01–10 课放在同一目录。

本篇以课堂笔记、`AI应用开发与测试实战.pdf`、`day09总结（langchain与langsmith）.xmind` 和配套数据文件为依据，整理 LangChain 组件、LangSmith 可观测性与评估，以及一个服装尺码 RAG 问答机器人的实现和测试方法。

> **阅读约定**：带有“课堂内容”的段落保留课堂主线；带有“整理补充”的段落是为了帮助测试、排错和落地而补充的实践建议。示例中的密钥、数据集名称和路径均为占位值，本笔记没有连接百炼或 LangSmith，也没有把示例代码的结果写成已验证结果。

## 一、学习目标与课程地图

完成本课后，应能：

1. 说明云端大模型 API、SDK 与应用框架之间的关系，并知道如何安全配置 API Key。
2. 解释 LangChain 1.x 中模型、工具、消息、记忆、流式传输、结构化输出和中间件的作用。
3. 用 Trace 理解一次 Agent 调用经过了哪些步骤，用数据集和评估器建立可重复的测试。
4. 拆解 RAG 问答机器人的加载、切分、向量化、检索、提示词注入和生成链路。
5. 从功能、可靠性、性能、安全和可观测性角度设计 Agent/RAG 测试，而不是只检查最终的一段文字。

课程主线：

```text
云端模型 API
    ↓
LangChain 组件化应用（Agent / Tool / Memory / RAG）
    ↓
LangSmith Trace 与 Evaluate
    ↓
服装尺码 RAG 问答机器人：构建 → 跟踪 → 数据集评估 → 定位问题
```

## 二、资料范围与版本提醒

### 2.1 本课资料

| 资料 | 用途 |
| --- | --- |
| `01-课堂资料/课堂笔记.md` | 课堂目标、LangChain/LangSmith 概念、示例代码和 RAG 评估流程。 |
| `01-课堂资料/AI应用开发与测试实战.pdf` | 26 页讲义，补充 LLMaaS、组件架构、测试维度、LangSmith 页面流程和 RAG 实现。 |
| `01-课堂资料/day09总结（langchain与langsmith）.xmind` | 课程总结文件，页面中确认存在，正文以课堂笔记和讲义为准。 |
| `01-课堂资料/AA-Omniscience_dataset_public.csv` | 课程目录中的数据文件；本篇只记录其存在，没有据文件名推断数据内容。 |
| `01-课堂资料/img` | 课堂截图目录，展示配置、Trace 和评估结果。 |

### 2.2 版本差异

课堂按 **LangChain 1.x** 讲解，核心入口使用 `create_agent()`。不同时间安装的 `langchain`、`langchain-openai`、`langchain-community`、`langchain-chroma` 和 `langsmith` 版本可能导致导入路径、参数名或返回对象结构不同。

- 新代码常用 `api_key`、`base_url`；旧代码可能写成 `openai_api_key`、`openai_api_base`。
- 消息可从 `langchain.messages` 导入，也可能需要从 `langchain_core.messages` 导入。
- 课堂示例中出现的 `ChatQwen`、`DashScopeEmbeddings` 等集成类，应以本地安装包和官方文档为准。
- 升级依赖后要重新运行最小调用、工具调用和评估样例，不能只看 `pip install` 成功。

## 三、前置：云端大模型与 API 调用

### 3.1 什么是 LLMaaS

云端大模型服务把已经训练好的模型部署在云端，开发者通过 API 按调用量付费，无需自建训练集群。课堂以阿里云百炼为例，也可迁移到其他兼容 OpenAI 协议的服务。

这种方式降低了模型运维门槛，但测试范围会扩大：除了答案质量，还要关注鉴权、配额、限流、网络超时、Token 消耗、模型版本变化和敏感数据出境。

### 3.2 课堂测试维度

| 维度 | 关注点 |
| --- | --- |
| 功能 | 不同用户输入下，响应是否完整、符合角色和业务规则。 |
| 性能 | 首 Token 延迟、总响应时间、吞吐量、并发下的错误率。 |
| 成本 | 输入/输出 Token、长上下文和重试造成的额外费用。 |
| 安全 | API Key 是否泄露，是否能抵抗提示词注入、越权工具调用和敏感信息回显。 |

> **整理补充**：测试报告应记录模型名称、供应商、区域、温度、最大输出 Token、提示词版本和时间窗口，否则同一用例的结果无法比较。

### 3.3 用环境变量保存密钥

不要把真实密钥写入 Notebook、源码仓库、日志或截图。下面只展示调用形态，不执行联网请求。

```python
import os
import dashscope

# 在操作系统或密钥管理服务中配置 DASHSCOPE_API_KEY
# 不要把真实值提交到代码仓库
api_key = os.getenv("DASHSCOPE_API_KEY")
if not api_key:
    raise RuntimeError("请先配置 DASHSCOPE_API_KEY")
dashscope.api_key = api_key


def call_qwen(user_question: str) -> str:
    response = dashscope.Generation.call(
        model="qwen-max",
        messages=[
            {"role": "system", "content": "你是一个只回答旅游问题的助手。"},
            {"role": "user", "content": user_question},
        ],
    )
    # 生产代码还要检查 HTTP/SDK 错误状态，再读取 output.text
    return response.output.text
```

运行前检查：SDK 版本、模型是否可用、密钥权限、配额和网络出口。不要把“代码可以解析”当作“模型调用已成功”。

## 四、LangChain 是什么

### 4.1 课堂定义

LangChain 是面向大语言模型应用的开源编程框架，通过模块化组件和工作流编排，把 LLM 与工具、私有数据和外部系统连接起来，便于构建 Agent、RAG 等应用。

原生 LLM 常见的四个落地痛点：

1. **没有持续记忆**：一次调用不会自动保留上一轮对话。
2. **没有实时联网能力**：无法直接获得训练截止时间后的信息。
3. **不了解企业私有数据**：不能直接读取本地文档、数据库和业务系统。
4. **难以形成业务流程**：单次文本生成不能自动完成多步骤、可审计的任务。

LangChain 通过组件化和编排缓解这些问题，但它不会自动保证结果正确。每个组件都需要自己的测试数据、失败处理和观测证据。

### 4.2 LangChain 1.x 生态

| 包/层 | 课堂定位 | 测试时要确认 |
| --- | --- | --- |
| `langchain-core` | 基础抽象和标准接口，像组件底座。 | 消息、Runnable、提示词和模型接口是否兼容。 |
| `langchain` | Agent、链和检索策略等主体框架。 | Agent 状态、工具循环、返回结构和版本。 |
| 集成包（如 `langchain-openai`） | 对接具体模型供应商。 | 参数、模型名、超时、重试和错误映射。 |
| `langchain-community` | 社区维护的加载器、Embedding 等集成。 | 依赖来源、维护状态和数据格式。 |
| `langgraph` | 编排复杂循环、分支和持久化状态。 | 节点、边、循环终止与恢复。 |
| `langsmith` | Trace、数据集、评估和生产监控。 | 项目名、追踪开关、数据脱敏和指标。 |

课堂安装路线示例：

```bash
pip install langchain langchain-core langchain-community
pip install langchain-openai langchain-chroma langchain-text-splitters
pip install langsmith langserve
```

按需安装集成包即可，避免为了“全家桶”引入不需要的依赖。安装后用 `pip list | findstr lang`（Windows）或 `pip list | grep lang`（Linux/macOS）记录实际版本。

## 五、LangChain 组件全景

课堂重点组件及整理补充的常见抽象如下：

| 组件 | 作用 | 典型测试问题 |
| --- | --- | --- |
| Prompt | 把角色、任务、约束和上下文组织成模型输入。 | 变量是否绑定、注入内容是否被当成指令、敏感规则是否泄露。 |
| Model | 生成或理解文本的推理引擎。 | 参数、超时、限流、稳定性和输出上限。 |
| Parser / Structured output | 把模型输出转成可校验的结构。 | JSON Schema、字段类型、缺失字段和截断。 |
| Chain / Runnable | 把多个步骤串成可组合的流程。 | 输入输出类型、异常传播、重试和幂等性。 |
| Agent | 根据任务决定是否调用工具并迭代完成任务。 | 选工具、参数、循环次数、越权和终止。 |
| Tool | 封装搜索、数据库或业务 API。 | 参数边界、权限、超时、重复执行和副作用。 |
| Message | 表示 system、human、AI、tool 等上下文单元。 | 角色顺序、tool_call_id、历史隔离和内容类型。 |
| Memory | 保存会话或业务状态。 | 跨用户串话、过期、清理、持久化和隐私。 |
| Retriever | 从文档或向量库召回相关内容。 | 召回率、排序、top-k、过滤器和空结果。 |
| Middleware | 在模型/工具前后插入限制、审批或脱敏逻辑。 | 是否能阻断危险操作，失败是否可审计。 |

## 六、Agent：模型、工具、记忆与规划

课堂用一个便于记忆的公式概括：

```text
Agent = LLM（大脑） + Tools（手脚） + Memory（记忆） + Planning（规划）
```

一次典型循环可以抽象为：

```text
用户目标
  ↓
模型分析并决定下一步
  ↓
需要工具？ ── 否 ──→ 生成最终回答
  │
  是
  ↓
生成工具名和参数 → 执行工具 → 把结果写回消息
  ↺ 再次分析，直到完成或达到上限
```

测试 Agent 时要观察中间步骤：

- 选择工具是否符合任务，而不是只看最终答案像不像对的。
- 工具参数是否完整、类型正确、没有把用户输入直接拼成危险命令。
- 工具异常、空结果和超时是否被模型如实处理。
- 循环是否有最大步数、总耗时和 Token 上限。
- 记忆是否按会话/用户隔离，是否会把旧任务带入新任务。

### 6.1 用 `create_agent` 创建最小 Agent

下面是课堂代码的 LangChain 1.x 写法。`ChatOpenAI` 通过 OpenAI 兼容接口调用百炼的 Qwen；参数名请按当前集成包确认。

```python
import os
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(
    model="qwen-max",
    api_key=os.environ["DASHSCOPE_API_KEY"],
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=0.2,
    timeout=60,
    max_tokens=1024,
    max_retries=2,
)

agent = create_agent(model=llm)

# 需要真实 API Key 和网络；本笔记不执行此调用
result = agent.invoke({"messages": [HumanMessage(content="你好，你是谁？")]})
print(result["messages"][-1].content)
```

**版本说明**：课堂示例使用 `openai_api_key`、`openai_api_base` 也可能仍能工作，但新旧参数不能混用后不做验证。若导入 `HumanMessage` 失败，尝试 `from langchain.messages import HumanMessage`。

## 七、模型（Models）

模型是应用的推理引擎。课堂重点解释静态模型：在一次执行中固定模型名称和参数；复杂流程也可以根据路由、成本或任务难度动态选择模型。

| 参数 | 含义 | 测试建议 |
| --- | --- | --- |
| `model` | 供应商侧模型名称。 | 记录精确名称和版本，检查模型下线/替换。 |
| `api_key` | 访问凭证。 | 使用密钥管理；日志中打码。 |
| `temperature` | 输出随机性。 | 事实问答和评估通常用较低值；比较时固定。 |
| `timeout` | 单次调用最长等待时间。 | 覆盖超时、重试和用户提示。 |
| `max_tokens` | 输出长度上限。 | 检查截断、JSON 不闭合和长答案质量。 |
| `max_retries` | SDK 层最大重试次数。 | 防止重复副作用，记录最终错误。 |

“更高温度更有创造力”只是趋势，不是质量保证；在测试中应通过多轮、固定数据和统计指标观察稳定性。

## 八、工具（Tools）与工具调用测试

工具让 Agent 能访问天气、搜索、数据库或业务 API。`@tool` 通常会根据函数签名和文档生成工具描述，模型再决定是否调用。

```python
from langchain.tools import tool

@tool
def weather_search(
    location: str,
    units: str = "celsius",
    include_forecast: bool = False,
) -> str:
    """查询指定地点的天气。仅返回模拟数据，用于演示工具接口。"""
    if not location.strip():
        raise ValueError("location 不能为空")
    if units not in {"celsius", "fahrenheit"}:
        raise ValueError("units 只允许 celsius 或 fahrenheit")

    temp = 22 if units == "celsius" else 72
    result = f"当前{location}天气：{temp}度"
    if include_forecast:
        result += "；未来5天：晴"
    return result

# 组装 Agent 时传入 tools=[weather_search]
```

课堂示例中的工具函数是模拟实现，不代表接入真实天气服务。测试工具时至少覆盖：

1. 正常地点、空地点、超长地点和特殊字符。
2. 默认参数、非法枚举、类型错误和缺失必填参数。
3. 工具超时、返回空值、返回错误对象和第三方限流。
4. 重复调用是否会产生重复扣款、写入或发送消息。
5. 工具返回内容含“忽略系统规则”时，模型是否把它当数据而不是新指令。

## 九、消息（Messages）

消息是模型上下文的基本单元，通常包含：

- **角色**：`system`、`human/user`、`ai/assistant`、`tool`。
- **内容**：文本，也可能是图片、音频或其他多模态内容。
- **元数据**：消息 ID、Token 使用量、响应信息等。

工具调用至少需要保持调用消息和结果消息的关联 ID。下面是结构示意：

```python
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
    ToolMessage,
)

messages = [
    SystemMessage(content="你是天气查询助手，只回答天气问题。"),
    HumanMessage(content="今天北京的天气？"),
    AIMessage(
        content="",
        tool_calls=[{
            "name": "weather_search",
            "args": {"location": "北京"},
            "id": "call_001",
        }],
    ),
    ToolMessage(
        content="北京今天晴，22度",
        tool_call_id="call_001",
    ),
]
```

测试消息链时，检查角色顺序、空内容、编码、多模态载荷、`tool_call_id` 和历史截断。不要只验证最后一条 `content`。

## 十、记忆、流式传输、结构化输出与中间件

### 10.1 记忆（Memory）

大模型不会自动记住上一次请求；框架通常把历史消息或状态重新传给模型。课堂区分：

- `InMemorySaver`：适合开发和测试，进程退出后通常丢失。
- 数据库持久化：适合生产，需要考虑过期、加密、清理和并发。

记忆测试重点是**隔离与生命周期**：A 用户不能读到 B 用户的历史；切换新会话后旧偏好不应无条件继承；达到上下文上限时应有可解释的截断策略。

### 10.2 流式传输（Streaming）

流式传输把输出片段逐步交给用户，改善首字节体验。测试要分别记录首 Token 延迟、片段顺序、断线重连、最终拼接结果和敏感词是否在中途泄露。流式“看起来更快”不等于总生成时间更短。

### 10.3 结构化输出（Structured output）

让模型按 JSON 或 Schema 返回，便于下游程序处理。测试应验证：合法 JSON、必填字段、数据类型、枚举、边界值、未知字段和截断时的失败路径。仅在提示词里写“请输出 JSON”不等于真正的 Schema 校验。

### 10.4 中间件（Middleware）

中间件是在模型或工具执行前后插入逻辑的插件机制。课堂示例包括：

- `HumanInTheLoopMiddleware`：敏感工具执行前人工审批。
- `ModelCallLimitMiddleware`：限制模型调用次数，避免无限循环。
- `PIIMiddleware`：检测和过滤输入、输出中的个人敏感信息。

整理补充：中间件必须记录“拦截原因、规则版本、决策人/决策结果”；只返回一段“操作失败”会让线上问题无法追溯。

### 10.5 组件交互示例：旅行规划 Agent

```text
用户：计划北京三日游
  ↓
模型识别任务并规划步骤
  ↓
调用天气工具 → 调用景点搜索工具 → 读取用户偏好记忆
  ↓
合并工具结果，检查缺失和冲突
  ↓
按结构化格式输出行程、依据和注意事项
```

可以把每一个箭头当成测试边界：工具是否按预期调用、失败是否可恢复、最终结果是否引用了实际工具返回值。复杂编排需要额外关注循环终止和并行工具调用的竞态。

## 十一、LangSmith 是什么

LangSmith 是面向 LLM 应用生命周期的平台，覆盖构建、调试、测试、监控和评估。课堂强调的实用价值是把 LangChain 应用从“黑盒”变成可追踪的执行过程：每次模型调用、工具调用、输入输出和耗时都能在 Trace 中查看。

| 能力 | 产出 | 测试价值 |
| --- | --- | --- |
| Trace / Debug | 每次运行的步骤树、输入输出、耗时和错误。 | 定位慢在哪一步、错在哪个工具或提示词。 |
| Dataset | 可复用的输入、参考输出和元数据集合。 | 建立回归基线，避免只凭手工体验判断。 |
| Evaluate | 对数据集批量运行目标函数和评估器。 | 比较模型、提示词、检索参数和版本。 |
| Monitor | 生产运行的延迟、错误和使用趋势。 | 发现发布后的退化和异常流量。 |

课堂资料提到个人 Develop 计划的免费 Trace 数量和套餐信息会变化，实际额度应以 LangSmith 当前页面为准，不要把课程截图中的数字当成长期承诺。

## 十二、配置 LangSmith 追踪

注册 `https://smith.langchain.com/` 后创建 API Key，再在运行环境配置：

```python
import os

# 仅写占位值；真实 API Key 应由密钥管理系统注入
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = os.environ.get("LANGSMITH_API_KEY", "<YOUR_API_KEY>")
os.environ["LANGSMITH_PROJECT"] = "day11-rag-demo"
```

也可以在操作系统层设置变量。Windows PowerShell 示例：

```powershell
$env:LANGSMITH_TRACING = "true"
$env:LANGSMITH_ENDPOINT = "https://api.smith.langchain.com"
$env:LANGSMITH_PROJECT = "day11-rag-demo"
# API Key 建议从安全存储读取，不要写进脚本或 Notebook
```

开启追踪后应先跑一个最小、无敏感数据的调用，确认项目名和数据脱敏策略，再接入完整业务。Trace 中可能包含用户原文、检索片段和工具参数，生产环境要按隐私要求配置采样、遮罩或禁用敏感字段。

## 十三、Trace 调试与慢响应定位

若 Agent 响应特别慢，可以按 Trace 的时间线定位，而不是直接把问题归因于“模型慢”：

1. 区分排队、网络连接、首 Token、完整生成和后处理时间。
2. 查看是否发生了多次模型调用、重复检索或 Agent 循环未及时终止。
3. 检查工具外部接口的 P95/P99 延迟、超时和重试次数。
4. 查看提示词和检索上下文是否异常变长，是否触发更多输出 Token。
5. 对比冷启动与热启动、串行与并行工具、不同模型和 `top_k`。
6. 把优化前后的 Trace 放在同一数据集上比较，不能只用一个“感觉更快”的样例。

建议记录的指标：总耗时、首 Token 延迟、模型调用次数、工具调用次数、输入/输出 Token、错误类型、重试次数和最终答案长度。

## 十四、创建评估数据集

数据集可以在网页中导入，也可以使用 LangSmith Client 创建。课堂示例采用“问题—标准答案”对；实际项目还可以加来源文档、业务标签、难度、版本和安全类别。

```python
import os
from langsmith import Client

# 需要真实 LangSmith API Key；本笔记不执行
client = Client(
    api_url=os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com"),
    api_key=os.environ["LANGSMITH_API_KEY"],
)

dataset = client.create_dataset(
    dataset_name="cloth-size-regression-v1",
    description="服装尺码 RAG 回归集；每条样例都应有可核验依据",
)

examples = [
    {
        "inputs": {"question": "身高163cm，体重96斤，选择多大尺码？"},
        "outputs": {"answer": "M码"},
    },
    {
        "inputs": {"question": "身高178cm，体重115斤，选择多大尺码？"},
        "outputs": {"answer": "L码"},
    },
]
client.create_examples(dataset_id=dataset.id, examples=examples)
```

**数据集质量要求**：参考答案应来自明确的尺码表或业务规则；对无法唯一确定的身材信息，应把“需要补充胸围/版型”等条件写入期望，而不是强行指定一个码。数据集变更要有版本号，避免评估基线悄悄改变。

## 十五、Evaluate：从精确匹配到可解释指标

课堂代码使用 `reference_content in output_content` 作为简单评估器，适合快速验证，但存在误判：例如输出“不是 M 码”也包含“M码”。整理后可至少做规范化、否定检查和证据检查，并把结果写入指标字典。

```python
import re


def normalize(text: str) -> str:
    return re.sub(r"\s+", "", str(text)).lower()


def exact_size_match(outputs: dict, reference_outputs: dict) -> dict:
    actual = normalize(outputs.get("answer", ""))
    expected = normalize(reference_outputs.get("answer", ""))
    if not expected:
        return {"key": "exact_size_match", "score": 0, "reason": "缺少参考答案"}

    # 这是示例规则，正式项目应根据业务定义否定、多个尺码和区间答案
    negative = f"不是{expected}" in actual or f"不选{expected}" in actual
    ok = expected in actual and not negative
    return {
        "key": "exact_size_match",
        "score": int(ok),
        "reason": "包含预期尺码且未检测到否定" if ok else "尺码或否定关系不满足",
    }
```

批量评估时还可以增加：

- **检索指标**：命中率、Recall@k、相关性、空检索率。
- **回答指标**：事实一致性、依据覆盖、拒答正确性、结构化格式通过率。
- **Agent 指标**：工具选择准确率、参数正确率、无谓调用率、循环步数。
- **系统指标**：P50/P95 延迟、错误率、Token/请求、成本和人工接管率。

一个“通过率 100%”只代表所选数据集和评估器下的结果，不能代表所有真实用户都正确。

## 十六、综合案例：服装尺码 RAG 问答机器人

### 16.1 项目背景

案例用服装属性构建本地知识库，用户输入身高、体重等信息后，系统检索尺码资料，交给模型生成个性化建议。用户可以更新本地知识，因此测试时必须验证“知识库版本”和“当前请求”是否对应。

### 16.2 RAG 原理

```text
本地文档
  ↓ 加载
切分为 chunks
  ↓ Embedding
向量数据库（Chroma 等）

用户问题
  ↓ Embedding / 相似检索
召回 top-k 文档片段
  ↓ 与问题拼入提示词
LLM 生成答案（可带依据和拒答）
```

RAG 提供“基于资料回答”的路径，但不自动保证召回正确。测试要分别验证切分、向量化、检索排序、上下文注入和最终生成，不能只看一个答案。

### 16.3 项目文件结构

| 文件 | 作用 |
| --- | --- |
| `cloth.txt` | 服装尺码等外部知识。 |
| `vectorstore.py` | 加载、切分、Embedding 并持久化 Chroma 向量库。 |
| `langchainRag.py` | 检索上下文、构造 Agent 并回答问题。 |
| `create_dataset.py` | 创建 LangSmith 数据集和样例。 |
| `eval_rag.py` | 封装目标函数、评估器并运行批量评估。 |

课堂示例使用 `chunk_size=50`、`chunk_overlap=10` 作为演示参数。它们不是通用最优值，正式项目应以检索实验和评估结果调参。

### 16.4 构建 Chroma 向量库

下面保留课堂流程，同时把 `persist_directory`、编码和重复写入风险写得更明确。`DashScopeEmbeddings` 需要对应集成包和 API Key。

```python
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import DashScopeEmbeddings

DATA_DIR = Path("data")
source = DATA_DIR / "cloth.txt"
db_dir = DATA_DIR / "chroma_cloth_db"

loader = TextLoader(str(source), encoding="utf-8")
documents = loader.load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    separators=["\n\n", "\n", "。", "，", " ", ""],
)
splits = splitter.split_documents(documents)

embeddings = DashScopeEmbeddings(model="text-embedding-v1")
vector_store = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="cloth-size-v1",
    persist_directory=str(db_dir),
)
print(f"写入 {len(splits)} 个文本块")
```

**幂等性提醒**：重复运行 `from_documents` 可能把相同文档写入多次。应使用稳定文档 ID、清理旧 collection，或先比较文件哈希和知识库版本。测试中要验证重建前后召回条数没有异常增长。

### 16.5 动态提示词与 RAG Agent

课堂用 `dynamic_prompt` 在每次请求前检索，把结果作为系统提示的一部分。整理时加上“资料不足时拒答”和分隔标记，降低文档内容被当成指令的风险。

```python
import os
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

# vector_store 应在前面加载完成
llm = ChatOpenAI(
    model="qwen-max",
    api_key=os.environ["DASHSCOPE_API_KEY"],
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

@dynamic_prompt
def prompt_with_context(request: ModelRequest) -> str:
    last_message = request.state["messages"][-1]
    query = getattr(last_message, "text", None) or str(last_message.content)
    retrieved_docs = vector_store.similarity_search(query, k=3)
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return (
        "你是服装尺码助手。只能依据<knowledge>中的资料回答；"
        "资料不足时说明需要补充的信息，不要猜测。\n"
        "<knowledge>\n" + context + "\n</knowledge>"
    )

agent = create_agent(model=llm, middleware=[prompt_with_context])

# 需要真实模型和向量库；本笔记不执行
result = agent.invoke({
    "messages": [HumanMessage(content="身高163cm，体重96斤，选择多大尺码？")]
})
print(result["messages"][-1].content)
```

版本差异：课堂示例使用 `request.state["messages"][-1].text`，不同版本消息对象可能没有 `.text`，需从 `.content` 读取；不要在未查看 Trace 和对象结构时直接复制粘贴。

## 十七、RAG 测试设计

### 17.1 分层检查

| 层次 | 例子 | 证据 |
| --- | --- | --- |
| 数据 | 文件编码、文档哈希、版本、重复记录。 | 加载日志、文件清单、哈希。 |
| 切分 | 块大小、重叠、表格/行的完整性。 | chunk 数量和抽样内容。 |
| 向量库 | Embedding 模型、collection、持久化目录。 | 配置、索引统计、重建对比。 |
| 检索 | top-k、相似度、过滤条件、空结果。 | 召回文档 ID、分数和原文片段。 |
| 提示词 | 上下文边界、角色规则、拒答条件。 | Trace 中的最终 prompt。 |
| 生成 | 尺码、依据、条件和格式。 | 参考答案、评估器、人工复核。 |

### 17.2 典型用例

- 资料中有明确尺码：应返回正确尺码并引用对应规则。
- 身高体重在两个区间之间：应说明版型/胸围等缺失信息，不能伪造精确结论。
- 查询完全不在服装知识范围：应按业务规则拒答或引导，而不是编造。
- 文档新增、更名、删除后：检索应反映当前版本，旧内容不能残留。
- 文档中含“忽略系统提示”的文字：应被当作知识内容，不改变系统规则。
- 向量库为空、Embedding 超时、模型不可用：应返回可识别的失败状态，不展示半份成功答案。

## 十八、LangSmith 评估的完整流程

课堂给出的综合流程可以落成以下八步：

1. **模型初始化**：加载 Embedding、向量库和对话模型。
2. **动态提示词**：定义检索并注入上下文的函数。
3. **构建 Agent**：把模型和中间件组合起来。
4. **目标函数**：把数据集输入转换为 Agent 调用，并返回统一字段。
5. **环境配置**：设置 LangSmith 追踪和项目名。
6. **创建 Client**：连接数据集和评估 API。
7. **定义评估器**：比较实际答案与参考答案，或计算检索/事实指标。
8. **执行 Evaluate**：批量运行，查看每条 Trace 和聚合指标。

```python
from langsmith import Client


def target(inputs: dict) -> dict:
    question = inputs["question"]
    result = agent.invoke({"messages": [HumanMessage(content=question)]})
    return {"answer": result["messages"][-1].content}

client = Client()
# 下面是调用形态，需以本地 langsmith 版本签名为准
# client.evaluate(
#     target,
#     data="cloth-size-regression-v1",
#     evaluators=[exact_size_match],
#     experiment_prefix="day11-rag-regression",
# )
```

执行后要保存实验名、代码/提示词版本、数据集版本和关键指标。评估异常时先打开单条 Trace，确认是目标函数、检索、模型还是评估器出了问题。

## 十九、围绕 LangChain 与 LangSmith 的测试清单

| 编号 | 场景 | 关键预期 |
| --- | --- | --- |
| LC-001 | API Key 缺失、过期或权限不足 | 启动或调用阶段给出明确错误，密钥不出现在日志。 |
| LC-002 | 模型名错误或服务不可用 | 受控失败/重试，不能返回伪造成功答案。 |
| LC-003 | 温度、Token 上限改变 | 配置实际生效，截断可识别。 |
| LC-004 | system/human/tool 角色顺序错误 | 进入失败或告警，不静默改变业务规则。 |
| LC-005 | 工具选择题 | 需要工具时调用正确工具，不需要时不调用。 |
| LC-006 | 工具参数为空、越界、类型错误 | 在工具边界拦截，不能把原始输入拼成危险命令。 |
| LC-007 | 工具超时或返回空值 | Agent 有限重试并解释失败，循环可终止。 |
| LC-008 | 有副作用的工具重复调用 | 具备幂等键/审批机制，避免重复写入或扣费。 |
| LC-009 | Agent 无限规划 | 达到步数、时长或 Token 上限并留下 Trace。 |
| LC-010 | 两个用户并发对话 | Memory 和工具权限严格隔离，不串话。 |
| LC-011 | 流式响应中途断开 | 服务端可取消/重连，最终状态不会伪装成完整成功。 |
| LC-012 | JSON 输出缺字段、错类型或截断 | Schema 校验失败可观测，下游不接收脏数据。 |
| LC-013 | PII 出现在输入、工具返回或 Trace | 按策略遮罩/拒绝/脱敏，日志可审计。 |
| LC-014 | Prompt injection 文档 | 文档内容只作为数据，不覆盖 system 规则。 |
| LC-015 | RAG 空库或召回无关片段 | 明确说明依据不足，不胡编答案。 |
| LC-016 | RAG top-k 和 chunk 参数变化 | 用固定检索集比较召回率和回答质量。 |
| LC-017 | 知识库更新/回滚 | Trace 显示当前版本，旧片段不意外残留。 |
| LC-018 | LangSmith 追踪开关关闭 | 业务仍可运行，且不会把敏感数据送到错误项目。 |
| LC-019 | Dataset 样例缺少参考答案 | 评估器返回可解释失败，不把缺失当通过。 |
| LC-020 | Evaluate 多轮回归 | 记录模型、提示词、依赖和数据集版本，比较指标变化。 |
| LC-021 | 评估器“包含匹配”遇到否定句 | 不把“不是 M 码”误判为通过。 |
| LC-022 | 生产延迟升高 | 结合 Trace 拆分模型、工具、检索和排队耗时。 |

## 二十、常见问题与排查

| 问题 | 排查顺序 |
| --- | --- |
| `create_agent` 或消息导入失败 | 记录包版本，查看当前版本文档；尝试 `langchain.messages` 与 `langchain_core.messages` 的对应路径。 |
| OpenAI 兼容接口报参数错误 | 确认 `api_key/base_url` 与旧版 `openai_api_key/openai_api_base` 没有混用，检查模型名。 |
| 工具没有被调用 | 看工具描述、函数签名和 Agent Trace；确认模型具备工具调用能力。 |
| 工具调用后不结束 | 设最大步数/超时，检查工具返回格式和终止条件。 |
| 记忆串话 | 检查 session/user key、并发写入和历史清理；用两个用户交叉回归。 |
| RAG 召回为空 | 检查文件路径、编码、Embedding Key、collection 名和持久化目录。 |
| 每次重跑文档数增加 | 重复写入向量库；清理 collection 或使用稳定 ID 和版本化目录。 |
| 答案脱离知识库 | 查看 Trace 的检索片段和最终 prompt；增加“仅依据资料/资料不足拒答”规则。 |
| LangSmith 没有 Trace | 变量名、项目名和 API Key 是否正确；确认运行环境真的读取到变量。 |
| Trace 含敏感信息 | 上线前配置脱敏、采样和访问权限；不要把生产原文直接复制到公共数据集。 |
| Evaluate 结果异常高/低 | 单条打开 Trace，确认 target 输出字段、评估器逻辑和参考答案版本。 |
| 生产延迟升高 | 结合 Trace 拆分模型、工具、检索和排队耗时。 |

不要把网络、依赖、模型、检索、评估器问题混成一个“模型不稳定”问题；先定位层次，再修复。

## 二十一、课后练习

1. 使用一个无敏感信息的小型 `cloth.txt`，记录文档哈希、切分数量和三个检索样例。
2. 为尺码机器人补充“不足信息拒答”和“文档中的提示词注入”用例。
3. 在相同数据集上比较 `top_k=1/3/5`，同时记录召回相关性和总延迟。
4. 把评估器从简单包含匹配改成“规范化 + 否定检查 + 证据字段”三项指标。
5. 故意让工具超时、返回空值和返回非法 JSON，观察 Agent 是否有限重试并产生可读 Trace。
6. 使用两个 session 并发对话，验证 Memory 不串话，并检查 Trace 是否包含不必要的个人信息。
7. 固定模型与依赖版本，修改提示词后重新运行 Evaluate，写出指标变化和失败样例。

## 资料来源与范围

- 网盘目录：`2026最新AI测试基础一套通关/2、AI智能体测试/day11-Langchain与langsmith`。
- `01-课堂资料/课堂笔记.md`：云端大模型、LangChain 组件、LangSmith 配置/跟踪/数据集/评估、服装尺码 RAG 案例。
- `01-课堂资料/AI应用开发与测试实战.pdf`：26 页讲义，包含 LLMaaS、四类测试维度、LangChain 生态、Agent 组件、LangSmith 核心能力和 RAG 流程。
- `01-课堂资料/day09总结（langchain与langsmith）.xmind`：课程总结文件，网盘目录中已核对存在。
- `01-课堂资料/AA-Omniscience_dataset_public.csv`：网盘目录中已核对存在，本篇未据文件名推断其正文。
- [百度网盘课程资料](https://pan.baidu.com/s/11X8sJa4vGgD_jhz4YZ95Ig?pwd=1234)，提取码 `1234`。

本篇没有实际调用百炼模型、写入 Chroma、上传 LangSmith 数据集或执行线上 Evaluate；代码用于学习和测试设计，运行前需要准备依赖、密钥、数据和网络环境。